In [ ]:
from google.colab import auth
auth.authenticate_user()

import pandas as pd
from datetime import datetime
import pytz
import time
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
from google.api_core.exceptions import NotFound, GoogleAPICallError

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
RUTA= "AUTOMATIZACION CONCILIACIONES"
DATASET_ID = "produccion"
TABLE_BASES_ID= "BASES_CONCILIACION_temp"
TABLE_TICKET_ID= "TICKET_INFORMACION_temp"
TABLE_ULTIMOS_PAGOS_ID= "ULTIMOS_PAGOS_temp"
lista = ['BASE INDIVIDUAL','TICKET INFO INDIVIDUAL']

In [ ]:
# @title
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)
zh_peru = pytz.timezone("America/Lima")

table_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_BASES_ID)
bigquery_client.delete_table(table_ref, not_found_ok=True)
table_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_TICKET_ID)
bigquery_client.delete_table(table_ref, not_found_ok=True)

schema_Bases = [
        bigquery.SchemaField("ID", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_OPERACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATE)
]

schema_Ticket_Info = [
        bigquery.SchemaField("IDEPOL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERT_BANCO_EXCEL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODPROD", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMPOL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CLIENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("STSCERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO_VIGENCIA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_VIGENCIA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_INI_ULT_COBERTURA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN_ULT_COBERTURA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("CAN_LQ_COB", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_INC", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_ANU", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("CAN_LQ_EMI", bigquery.enums.SqlTypeNames.INT64),
        bigquery.SchemaField("DOC_COB", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DOC_ESTADO_COB", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_COB", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_VE_COB", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DOC_ANU_INC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DOC_ESTADO_ANU_INC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_INC_ANU", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("FECHA_VE_INCANU", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("PRIMA_BRUTA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("TIPODOC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO_LA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("ESTADO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CAN_LA_EMI", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMSIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("STSSIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OCURR", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("MTOTOTRES", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTAPROB", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTPAGADO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MTOTOTPEND", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("DESCRAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPOVIG", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODMONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CALCULO_PRIMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_CLIENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_CLIENTE_FACTURAR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CLIENTE_RESPONSABLE_PAGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.STRING)
]


def mover_archivo(origen, destino):
    bucket.copy_blob(origen, bucket, new_name=destino )
    origen.delete()
    return

def restar_fecha(fecha, periodicidad):
    if pd.isna(fecha):
        return fecha
    if periodicidad == 'M-MENSUAL':
        return fecha - pd.DateOffset(months=1)
    elif periodicidad in ('A-ANUAL','E-ESPECIAL'):
        return fecha - pd.DateOffset(years=1)
    else:
        return fecha

def normalizar_dataframe(df, schema, tipo):
    columnas = [field.name for field in schema]
    for col in columnas:
        if col not in df.columns:
            if tipo == 1:
                if col in ['FECHA_ALTA','FECHA_BAJA','FECHA_OPERACION','FECHA_CIERRE']:
                    df[col]= pd.Series(pd.NaT, dtype="datetime64[ns]")
                else:
                    df[col] = None
            if tipo == 2:
                if col in ['FECHA_INICIO_VIGENCIA','FECHA_FIN_VIGENCIA','FECHA_INI_ULT_COBERTURA',
                           'FECHA_FIN_ULT_COBERTURA','FECHA_VE_COB','FECHA_VE_INCANU']:
                    df[col]= pd.Series(pd.NaT, dtype="datetime64[ns]")
                else:
                    df[col] = None
            if tipo == 3:
                if col in ['FECHA_OPERACION']:
                    df[col]= pd.Series(pd.NaT, dtype="datetime64[ns]")
                else:
                    df[col] = None
    return df[columnas]


def renombrar_col_duplicadas(columnas):
    contador = {}
    nuevas_columnas = []
    for col in columnas:
        if col in contador:
            contador[col] += 1
            nuevas_columnas.append(f"{col}_{contador[col]}")
        else:
            contador[col] = 1
            nuevas_columnas.append(col)
    return nuevas_columnas

#### FUNCION PARA COMPROBAR SI LA TABLA ESTA CREADA
def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref, schema=schema)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        job_config.autodetect = False
        print(f'   ℹ️ ... Se ha creado la tabla: {table_id} en el dataset: {dataset_id} ...')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    return

def Ejecutar_Query(opcion):
    tabla_bases = bigquery_client.dataset(DATASET_ID).table(TABLE_BASES_ID)
    tabla_ticket_info = bigquery_client.dataset(DATASET_ID).table(TABLE_TICKET_ID)
    tabla_ultimos_pagos = bigquery_client.dataset(DATASET_ID).table(TABLE_ULTIMOS_PAGOS_ID)
    if opcion== 1:
        print(f"⏳ Buscando Ultimos pagos en tramas diarias ....")
        query = f"""
            CREATE OR REPLACE TABLE `{tabla_ultimos_pagos}` AS
            WITH filtro_tramas AS (
              SELECT CERTIFICADO, TIPO_DE_SEGURO, FECHA_DE_INICIO_DEL_SEGURO,
              FECHA_FIN_DEL_SEGURO, PERIODO_DE_PAGO, MONEDA, PRIMA_BRUTA, FECHA_DECLARADA,
              CONCAT(SUBSTR(CERTIFICADO, 1,4), SUBSTR(CERTIFICADO, 7,2), SUBSTR(CERTIFICADO, 11,10)) AS CERT_ID
              FROM (
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2018`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2019`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2020`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2021`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2022`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2023`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2024`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2025`
                    UNION ALL
                    SELECT * FROM `produccion.TRAMAS_diarias_BBVA_2026`
                  )
              WHERE TIPO_DE_MOVIMIENTO = '4' AND TIPO_DE_REGISTRO = '01'
            ),
            cruces AS (
              SELECT b.*, t.CERTIFICADO, t.TIPO_DE_SEGURO AS COD_PRODUCTO_TRAMA, PERIODO_DE_PAGO,
                  DATE(t.FECHA_DE_INICIO_DEL_SEGURO) AS FECHA_INICIO, DATE(T.FECHA_FIN_DEL_SEGURO) AS FECHA_FIN,
                  PRIMA_BRUTA, DATE(FECHA_DECLARADA) AS FECHA_RECEPCION,
                  MIN(DATE(t.FECHA_DE_INICIO_DEL_SEGURO)) OVER  (PARTITION BY CERTIFICADO_BANCO ) AS FECHA_INICIO_1ER_PAGO,
                  ROW_NUMBER() OVER  (PARTITION BY CERTIFICADO_BANCO ORDER BY t.FECHA_DE_INICIO_DEL_SEGURO DESC) AS NRO,
                  t.CERT_ID
              FROM `{tabla_bases}` b
              LEFT JOIN filtro_tramas t ON CONCAT(SUBSTR(b.CERTIFICADO_BANCO, 1,4),
                                            SUBSTR(b.CERTIFICADO_BANCO, 7,2), SUBSTR(b.CERTIFICADO_BANCO, 11,10)) = t.CERT_ID
            )
            SELECT CERTIFICADO_BANCO, CERT_ID, c.CERTIFICADO AS CERTIFICADO_TRAMA, c.COD_PRODUCTO_TRAMA, e.PRODUCTO, PRIMA_BRUTA,
                  PERIODO_DE_PAGO, FECHA_INICIO, FECHA_FIN, FECHA_INICIO_1ER_PAGO, FECHA_RECEPCION,
            FROM cruces c
            LEFT JOIN `produccion.EQUIVALENCIAS_PRODUCTOS` e ON c.COD_PRODUCTO_TRAMA= e.CODIGO_PRODUCTO
            WHERE NRO= 1
        """
        result = bigquery_client.query(query)
        result.result()
        print(f"✅ Ultimos pagos Completado ....")
        return
    if opcion== 2:
        print(f"⏳ Analizando informacion ....")
        query = f"""
            WITH concat_ticket_info AS (
                SELECT *,
                    CASE
                        WHEN TIPO = 'ACSELX' THEN
                            CONCAT(CODPROD, '-', NUMPOL, '-',NUMCERT)
                        ELSE
                            CONCAT(CODPROD, '-', NUMPOL)
                    END AS CONCATENADO,
                    CONCAT(SUBSTR(CERTIFICADO_BANCO, 1,4), SUBSTR(CERTIFICADO_BANCO, 7,2), SUBSTR(CERTIFICADO_BANCO, 11,10)) AS CERT_ID
                FROM (
                        SELECT DISTINCT *
                        FROM `{tabla_ticket_info}`
                    )
            ),
            nros_las AS (
                SELECT
                CONCATENADO,
                COUNT(NRO_LA) AS CANT_LA,
                SUM(MONTO) AS SUM_MONTOS_LA,
                STRING_AGG(CAST(NRO_LA AS STRING), ', ') AS CONCAT_NROS_LA,
                STRING_AGG(CAST(MONTO AS STRING), ', ') AS CONCAT_MONTOS,
                STRING_AGG(CAST(ESTADO_ACTUAL AS STRING), ', ') AS CONCAT_ESTADOS,
                STRING_AGG(CAST(NUMOPER AS STRING), ', ') AS CONCAT_NUM_OPER
              FROM `produccion.REPORTE_LA`
              GROUP BY CONCATENADO
            ),
            resumen AS(
                SELECT B.CERTIFICADO_BANCO, TRIM(T.CODPROD) AS COD_PROD, B.MONEDA, B.PRIMA AS MONTO_DEVOLVER, T.IDEPOL, T.CONCATENADO,
                    U.PRODUCTO, T.STSCERT, T.FECHA_INICIO_VIGENCIA, T.FECHA_INI_ULT_COBERTURA, T.FECHA_FIN_ULT_COBERTURA,
                    U.FECHA_INICIO AS FECHA_INICIO_ULT_PAGO,
                    T.DOC_ANU_INC, T.DOC_ESTADO_ANU_INC, T.PRIMA_INC_ANU,
                    T.PRIMA_COB, T.CAN_LQ_COB, T.CAN_LQ_INC, T.CAN_LQ_ANU, T.CAN_LQ_EMI,
                    ROUND((PRIMA_COB * CAN_LQ_COB),2) AS PRIMA_TOTAL,
                    ROUND((PRIMA - (PRIMA_COB * CAN_LQ_COB)),2) AS DIFERENCIA_TOTAL,
                    (PRIMA - L.SUM_MONTOS_LA) AS DIF_LA,
                    CASE
                        WHEN T.CODMONEDA = 'SOL' THEN 'PEN'
                        ELSE T.CODMONEDA
                    END AS MONEDA_TICKET,
                    CASE
                        WHEN CANT_LA IS NOT NULL THEN CANT_LA
                        ELSE 0
                    END AS CANTIDAD_LA,
                    CASE
                        WHEN SUM_MONTOS_LA IS NOT NULL THEN ROUND(SUM_MONTOS_LA,2)
                        ELSE 0
                    END AS SUMA_MONTO_LA,
                    CONCAT_NROS_LA, CONCAT_MONTOS, CONCAT_ESTADOS, CONCAT_NUM_OPER,
                    T.STSSIN, B.FECHA_OPERACION, CALCULO_PRIMA,
                    B.FECHA_OPERACION BETWEEN T.FECHA_INICIO_VIGENCIA AND T.FECHA_FIN_ULT_COBERTURA AS EN_COBERTURA,
                    ROW_NUMBER() OVER ( PARTITION BY B.CERTIFICADO_BANCO, B.FECHA_OPERACION
                        ORDER BY
                            CASE
                                WHEN B.FECHA_OPERACION BETWEEN T.FECHA_INICIO_VIGENCIA AND T.FECHA_FIN_ULT_COBERTURA
                                    THEN 1
                                ELSE 2
                            END,
                            T.FECHA_INICIO_VIGENCIA DESC
                    ) AS NRO_REG
                FROM `{tabla_bases}` B
                LEFT JOIN concat_ticket_info T ON CONCAT(SUBSTR(B.CERTIFICADO_BANCO, 1,4), SUBSTR(B.CERTIFICADO_BANCO, 7,2),
                                                    SUBSTR(B.CERTIFICADO_BANCO, 11,10))= T.CERT_ID

                LEFT JOIN nros_las L ON T.CONCATENADO = L.CONCATENADO
                LEFT JOIN `{tabla_ultimos_pagos}` U ON
                            CONCAT(SUBSTR(B.CERTIFICADO_BANCO, 1,4), SUBSTR(B.CERTIFICADO_BANCO, 7,2),
                            SUBSTR(B.CERTIFICADO_BANCO, 11,10))= U.CERT_ID

            ),
            Detalles AS(
                SELECT CERTIFICADO_BANCO, MONEDA, FECHA_OPERACION, MONTO_DEVOLVER, IDEPOL, CONCATENADO,
                        PRODUCTO, STSCERT, FECHA_INICIO_VIGENCIA, DATE(FECHA_INI_ULT_COBERTURA) AS FECHA_INI_ULT_COBERTURA,
                        FECHA_FIN_ULT_COBERTURA, FECHA_INICIO_ULT_PAGO,
                        DOC_ANU_INC, DOC_ESTADO_ANU_INC, PRIMA_INC_ANU,
                        PRIMA_COB, CAN_LQ_COB, CAN_LQ_INC, CAN_LQ_ANU, CAN_LQ_EMI,
                        PRIMA_TOTAL, DIFERENCIA_TOTAL, CONCAT_NROS_LA,
                        CONCAT_MONTOS, SUMA_MONTO_LA, ROUND(DIF_LA,2) AS DIF_LA,
                        CONCAT_ESTADOS, CANTIDAD_LA, CONCAT_NUM_OPER, STSSIN, EN_COBERTURA, MONEDA_TICKET, CALCULO_PRIMA,
                        CASE
                            WHEN MONEDA != MONEDA_TICKET THEN 'ERROR EN MONEDA'
                            WHEN CANTIDAD_LA > 0 AND CONCAT_ESTADOS LIKE '%ACT%' THEN
                                CASE
                                    WHEN MONEDA = 'PEN' AND DIF_LA BETWEEN -1 AND 1 THEN 'ENVIAR A COBRANZAS'
                                    WHEN MONEDA = 'USD' AND DIF_LA BETWEEN -0.5 AND 0.5 THEN 'ENVIAR A COBRANZAS'
                                    WHEN MONEDA = 'PEN' AND DIF_LA NOT BETWEEN -1 AND 1 AND
                                                    COD_PROD IN ('1355','1365','1403','1422') THEN 'POLIZA CON REASEGURO'
                                    WHEN MONEDA = 'USD' AND DIF_LA NOT BETWEEN -0.5 AND 0.5 AND
                                                    COD_PROD IN ('1355','1365','1403','1422') THEN 'POLIZA CON REASEGURO'
                                    WHEN MONEDA = 'PEN' AND DIF_LA NOT BETWEEN -1 AND 1 THEN
                                        CASE
                                            WHEN PRIMA_TOTAL >= MONTO_DEVOLVER THEN 'AJUSTAR LA'
                                            ELSE "FALTA MONTO EMITIDO"
                                        END
                                    WHEN MONEDA = 'USD' AND DIF_LA NOT BETWEEN -0.5 AND 0.5 THEN
                                        CASE
                                            WHEN PRIMA_TOTAL >= MONTO_DEVOLVER THEN 'AJUSTAR LA'
                                            ELSE "FALTA MONTO EMITIDO"
                                        END
                                END
                            WHEN CANTIDAD_LA > 0 AND CONCAT_ESTADOS LIKE '%PAG%' THEN 'LA PAGADA'
                            WHEN CANTIDAD_LA > 0 AND CONCAT_ESTADOS LIKE '%OPT%' THEN 'LA OPT'
                            WHEN CANTIDAD_LA > 0 AND CONCAT_ESTADOS LIKE '%CMP%' THEN 'LA CMP'
                            WHEN CANTIDAD_LA > 0 AND CONCAT_ESTADOS LIKE '%PGP%' THEN 'LA PGP'

                            WHEN MONTO_DEVOLVER <= PRIMA_TOTAL THEN
                                CASE
                                    WHEN STSCERT = 'ACT' AND CANTIDAD_LA = 0 AND CAN_LQ_INC = 0 AND
                                                        CAN_LQ_ANU > 0 THEN 'ANULACION CON LQ ANULADAS'

                                    WHEN STSCERT = 'ANU' AND CANTIDAD_LA = 0 AND CAN_LQ_INC = 0 AND
                                                        CAN_LQ_ANU > 0 THEN 'REHABILITACION CON LQ ANULADAS'

                                    WHEN STSCERT IN ('ACT','ANU') AND CAN_LQ_INC > 0 AND CANTIDAD_LA = 0
                                                                    AND CAN_LQ_COB= 0 AND CAN_LQ_ANU = 0 THEN 'PAGOS EN INC'

                                    WHEN STSCERT = 'ACT' AND CANTIDAD_LA = 0 AND CAN_LQ_INC = 0 AND CAN_LQ_ANU = 0 THEN 'ANULACION'

                                    WHEN STSCERT = 'ANU' AND CANTIDAD_LA = 0 AND CAN_LQ_INC = 0 AND CAN_LQ_ANU = 0 THEN 'REHABILITACION'

                                    WHEN STSCERT = 'REN' AND CANTIDAD_LA = 0 AND CAN_LQ_INC = 0 AND CAN_LQ_ANU = 0 THEN 'REN - ACT'

                                    WHEN STSCERT = 'ACT' AND CANTIDAD_LA = 0 AND CAN_LQ_INC > 0 AND
                                                                            CAN_LQ_ANU >= 0 THEN 'ANULACION CON LQ ANULADAS Y LQ INC'
                                    WHEN STSCERT = 'ANU' AND CANTIDAD_LA = 0 AND CAN_LQ_INC > 0 AND
                                                                            CAN_LQ_ANU >= 0 THEN 'REHABILITACION CON LQ ANULADAS Y LQ INC'
                                    WHEN STSCERT = 'REN' AND CANTIDAD_LA = 0 AND CAN_LQ_INC > 0 AND
                                                                            CAN_LQ_ANU >= 0 THEN 'REN - ACT CON LQ ANULADAS Y LQ INC'

                                END
                            WHEN STSCERT = 'VAL' THEN 'POLIZA ESTADO VAL'
                            WHEN STSCERT = 'EXC' THEN 'POLIZA ESTADO EXCLUIDO'

                            WHEN FECHA_INICIO_ULT_PAGO > FECHA_FIN_ULT_COBERTURA  THEN 'PENDIENTE EMITIR'

                            WHEN STSCERT= 'ACT' AND PRIMA_INC_ANU >= MONTO_DEVOLVER AND CANTIDAD_LA = 0 AND CAN_LQ_ANU = 0 AND
                                                                            PRIMA_COB IS NULL THEN 'PENDIENTE APLICAR PAGOS'

                            WHEN STSCERT= 'ANU' AND PRIMA_INC_ANU >= MONTO_DEVOLVER AND CANTIDAD_LA = 0 AND CAN_LQ_ANU = 0 AND
                                                                            PRIMA_COB IS NULL THEN 'REVERSAR ANULACION - APLICAR PAGOS'

                            WHEN STSCERT= 'ANU' AND PRIMA_INC_ANU >= MONTO_DEVOLVER AND CANTIDAD_LA = 0 AND CAN_LQ_ANU > 0 AND
                                                                            CAN_LQ_INC = 0  AND PRIMA_COB IS NULL THEN 'REHABILITAR Y APLICAR PAGOS'

                            WHEN STSCERT= 'ACT' AND PRIMA_INC_ANU >= MONTO_DEVOLVER AND CANTIDAD_LA = 0 AND CAN_LQ_ANU > 0 AND
                                                                PRIMA_COB IS NULL THEN 'ANULAR INICO DE VIG - REHABILITAR Y APLICAR PAGOS'

                            WHEN MONTO_DEVOLVER > PRIMA_TOTAL OR MONTO_DEVOLVER > PRIMA_INC_ANU THEN 'FALTA MONTO EMITIDO'
                            WHEN STSSIN IS NOT NULL THEN 'POLIZA CON SINIESTRO'
                            WHEN IDEPOL IS NULL THEN 'NO SE ENCONTRO EN TICKET INFO'
                        END AS DETALLE
                FROM resumen
                WHERE NRO_REG= 1 OR IDEPOL IS NULL
            )
            SELECT *,
                CASE
                    WHEN DETALLE IN ('AJUSTAR LA', 'LA CMP', 'LA OPT',
                                        'LA PAGADA','LA PGP', 'POLIZA CON REASEGURO') THEN 'REVISIÓN PREVIA LAS'

                    WHEN DETALLE IN ('ANULACION CON LQ ANULADAS',
                                        'ANULACION CON LQ ANULADAS Y LQ INC',
                                        'ANULAR INICO DE VIG - REHABILITAR Y APLICAR PAGOS',
                                        'REHABILITACION CON LQ ANULADAS Y LQ INC',
                                        'REHABILITAR Y APLICAR PAGOS',
                                        'REN - ACT CON LQ ANULADAS Y LQ INC',
                                        'REVERSAR ANULACION - APLICAR PAGOS') THEN 'REVISIÓN PREVIA PARA PROCESAR'

                    WHEN DETALLE IN ('ANULACION', 'ENVIAR A COBRANZAS', 'PAGOS EN INC',
                                        'PENDIENTE APLICAR PAGOS','REHABILITACION', 'REN - ACT') THEN 'LISTOS PARA PROCESAR'

                    WHEN DETALLE IN ('FALTA MONTO EMITIDO', 'PENDIENTE EMITIR', 'POLIZA ESTADO VAL') THEN 'REVISIÓN PREVIA - EMISIÓN'

                    WHEN DETALLE IN ('ERROR EN MONEDA', 'POLIZA CON SINIESTRO', 'POLIZA ESTADO EXCLUIDO') THEN 'DETALLES PARTICULARES'

                    WHEN DETALLE = 'NO SE ENCONTRO EN TICKET INFO' THEN 'INFO NO UBICADA'
                END AS RESUMEN
            FROM Detalles
        """
        df = bigquery_client.query(query).to_dataframe()
        print(f"✅ Analisis Completado  ....")
        return df


def procesar_archivo(lista_archivos, carpeta):
    for blob in lista_archivos:
      if blob.name.endswith(".xlsx"):
        nombre_archivo = os.path.basename(blob.name)
        file = blob.download_as_string()
        if carpeta == 'BASE INDIVIDUAL':
            print(f"   ⏳ ... CARGANDO ARCHIVO: {nombre_archivo} ...")
            df_datos= pd.read_excel(BytesIO(file), sheet_name=0, dtype=str)
            df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

            df_datos= df_datos.rename(columns={'CERTIFICADO':'CERTIFICADO_BANCO','MONTO':'PRIMA'})
            #df_datos['PRODUCTO'] = df_datos['PRODUCTO'].str.upper()
            df_datos['PRIMA'] = df_datos['PRIMA'].replace({',': ''}, regex=True)
            df_datos['PRIMA'] = pd.to_numeric(df_datos['PRIMA'], errors="coerce").astype('float64')
            df_datos['FECHA_OPERACION'] = pd.to_datetime(df_datos['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos.loc[df_datos['MONEDA'].fillna('').str.strip().str.lower().isin(['soles','sol']), 'MONEDA'] = 'PEN'
            df_datos.loc[df_datos['MONEDA'].fillna('').str.strip().str.lower().isin(['dólares', 'dolares']), 'MONEDA'] = 'USD'
            df_datos= df_datos[['ID','CERTIFICADO_BANCO','MONEDA','PRIMA','FECHA_OPERACION']]

            fecha_carga = datetime.now(zh_peru).date()
            df_datos['NOMBRE_ARCHIVO']= nombre_archivo
            df_datos['FECHA_CARGA'] = fecha_carga
            df_datos = normalizar_dataframe(df_datos, schema_Bases, 1)

            ##### Guardar dataframes en BigQuery
            Guardar_en_BigQuery(df_datos, DATASET_ID, TABLE_BASES_ID, schema_Bases)
            print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")

        if carpeta == 'TICKET INFO INDIVIDUAL':
            print(f"   ⏳ ... CARGANDO ARCHIVO: {nombre_archivo} ...")
            df_datos= pd.read_excel(BytesIO(file), sheet_name=0, dtype=str)
            df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))
            df_datos= df_datos.rename(columns={'NRO_CERT_BANCO':'CERTIFICADO_BANCO', 'TIPO_SEGURO':'PRODUCTO',
                                'FEC_INI_VIG_CERT':'FECHA_INICIO_VIGENCIA', 'FEC_FIN_VIG_CERT':'FECHA_FIN_VIGENCIA',
                                'FECVE_COB':'FECHA_VE_COB', 'FECVE_COB_1':'FECHA_VE_INCANU','FECOCURR':'FECHA_OCURR',
                                'DOCUMENTO_COB':'DOC_COB', 'DOCUMENTO_ESTADO_COB':'DOC_ESTADO_COB',
                                'DOCUMENTO_ANU_INC':'DOC_ANU_INC','DOCUMENTO_ESTADO_ANU_INC':'DOC_ESTADO_ANU_INC',
                                'P_BRUTA':'PRIMA_BRUTA','MTO_LA':'MONTO_LA'})
            df_datos['FECHA_INICIO_VIGENCIA'] = pd.to_datetime(df_datos['FECHA_INICIO_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos['FECHA_FIN_VIGENCIA'] = pd.to_datetime(df_datos['FECHA_FIN_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos['FECHA_INI_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_INI_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos['FECHA_FIN_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_FIN_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos['FECHA_VE_COB'] = pd.to_datetime(df_datos['FECHA_VE_COB'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos['FECHA_VE_INCANU'] = pd.to_datetime(df_datos['FECHA_VE_INCANU'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos['FECHA_OCURR'] = pd.to_datetime(df_datos['FECHA_OCURR'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
            df_datos['CAN_LQ_COB'] = pd.to_numeric(df_datos['CAN_LQ_COB'], errors='coerce').astype('Int64')
            df_datos['CAN_LQ_INC'] = pd.to_numeric(df_datos['CAN_LQ_INC'], errors='coerce').astype('Int64')
            df_datos['CAN_LQ_ANU'] = pd.to_numeric(df_datos['CAN_LQ_ANU'], errors='coerce').astype('Int64')
            df_datos['CAN_LQ_EMI'] = pd.to_numeric(df_datos['CAN_LQ_EMI'], errors='coerce').astype('Int64')
            df_datos['PRIMA_COB'] = pd.to_numeric(df_datos['PRIMA_COB'], errors="coerce").astype('float64')
            df_datos['PRIMA_INC_ANU'] = pd.to_numeric(df_datos['PRIMA_INC_ANU'], errors="coerce").astype('float64')
            df_datos['PRIMA_BRUTA'] = pd.to_numeric(df_datos['PRIMA_BRUTA'], errors="coerce").astype('float64')
            df_datos['MONTO_LA'] = pd.to_numeric(df_datos['MONTO_LA'], errors="coerce").astype('float64')
            df_datos['MTOTOTRES'] = pd.to_numeric(df_datos['MTOTOTRES'], errors="coerce").astype('float64')
            df_datos['MTOTOTAPROB'] = pd.to_numeric(df_datos['MTOTOTAPROB'], errors="coerce").astype('float64')
            df_datos['MTOTOTPAGADO'] = pd.to_numeric(df_datos['MTOTOTPAGADO'], errors="coerce").astype('float64')
            df_datos['MTOTOTPEND'] = pd.to_numeric(df_datos['MTOTOTPEND'], errors="coerce").astype('float64')

            df_datos['FECHA_INI_ULT_COBERTURA'] = df_datos.apply(lambda r: restar_fecha(r['FECHA_FIN_ULT_COBERTURA'], r['CALCULO_PRIMA']), axis=1)

            fecha_carga = datetime.now(zh_peru).date()
            df_datos['NOMBRE_ARCHIVO']= nombre_archivo
            df_datos['TIPO'] = 'ACSELX'
            df_datos['FECHA_CARGA']= fecha_carga
            df_datos = normalizar_dataframe(df_datos, schema_Ticket_Info,2) # Normalizar la tabla

            Guardar_en_BigQuery(df_datos, DATASET_ID, TABLE_TICKET_ID, schema_Ticket_Info)
            print(f"   ✅  ARCHIVO - {nombre_archivo} - CARGADO ...")

        destino= f"{RUTA}/{carpeta}/PROCESADOS/{nombre_archivo}"
        mover_archivo(blob, destino)
    return

############### PROCESO DE CARGA DE ARCHIVOS EXCEL A BIGQUERY ##########
bucket = storage_client.bucket(BUCKET_NAME)

for folder in lista:
    ruta= f"{RUTA}/{folder}/NUEVOS"
    blobs_excels = list(bucket.list_blobs(prefix= ruta))
    print(f"ℹ️ CARPETA {folder}:")
    procesar_archivo(blobs_excels, folder)
Ejecutar_Query(1)
df_resultados= Ejecutar_Query(2)
fecha_hora = datetime.now(zh_peru).strftime("%Y-%m-%d_%H%M%S")
archivo_analisis= f"/content/analisis_conciliaciones_{fecha_hora}.xlsx"
df_resultados.to_excel(archivo_analisis, index=False)
print(f"💾 Archivo generado: {archivo_analisis}")


ℹ️ CARPETA BASE INDIVIDUAL:
   ⏳ ... CARGANDO ARCHIVO: Analisis_para_ticket_1417899.xlsx ...
   ℹ️ ... Se ha creado la tabla: BASES_CONCILIACION_temp en el dataset: produccion ...
   ✅  ARCHIVO - Analisis_para_ticket_1417899.xlsx - CARGADO ...
ℹ️ CARPETA TICKET INFO INDIVIDUAL:
   ⏳ ... CARGANDO ARCHIVO: 1417899.xlsx ...
   ℹ️ ... Se ha creado la tabla: TICKET_INFORMACION_temp en el dataset: produccion ...
   ✅  ARCHIVO - 1417899.xlsx - CARGADO ...
⏳ Buscando Ultimos pagos en tramas diarias ....
✅ Ultimos pagos Completado ....
⏳ Analizando informacion ....
✅ Analisis Completado  ....
💾 Archivo generado: /content/analisis_conciliaciones_2026-06-15_134802.xlsx


## PRUEBAS

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
ruta= f"{RUTA}/{lista[1]}/NUEVO"
blob_list = list(bucket.list_blobs(prefix=ruta))
for blob in blob_list:
    if blob.name.endswith(".xlsx"):
      print(f"ARCHIVO: {blob.name}")

ARCHIVO: AUTOMATIZACION CONCILIACIONES/TICKET INFO INDIVIDUAL/NUEVO/1394635.xlsx


In [ ]:
file = blob.download_as_string()
df_datos= pd.read_excel(BytesIO(file), sheet_name=0, dtype=str)
df_datos.columns = (df_datos.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

In [ ]:
df_datos= df_datos.rename(columns={'NRO_CERT_BANCO':'CERTIFICADO_BANCO', 'TIPO_SEGURO':'PRODUCTO',
                                  'FEC_INI_VIG_CERT':'FECHA_INICIO_VIGENCIA', 'FEC_FIN_VIG_CERT':'FECHA_FIN_VIGENCIA',
                                  'FECVE_COB':'FECHA_VE_COB', 'FECVE_COB_1':'FECHA_VE_INCANU','FECOCURR':'FECHA_OCURR',
                                  'DOCUMENTO_COB':'DOC_COB', 'DOCUMENTO_ESTADO_COB':'DOC_ESTADO_COB',
                                  'DOCUMENTO_ANU_INC':'DOC_ANU_INC','DOCUMENTO_ESTADO_ANU_INC':'DOC_ESTADO_ANU_INC',
                                  'P_BRUTA':'PRIMA_BRUTA','MTO_LA':'MONTO_LA'})
df_datos['FECHA_INICIO_VIGENCIA'] = pd.to_datetime(df_datos['FECHA_INICIO_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_datos['FECHA_FIN_VIGENCIA'] = pd.to_datetime(df_datos['FECHA_FIN_VIGENCIA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_datos['FECHA_INI_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_INI_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_datos['FECHA_FIN_ULT_COBERTURA'] = pd.to_datetime(df_datos['FECHA_FIN_ULT_COBERTURA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_datos['FECHA_VE_COB'] = pd.to_datetime(df_datos['FECHA_VE_COB'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_datos['FECHA_VE_INCANU'] = pd.to_datetime(df_datos['FECHA_VE_INCANU'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_datos['FECHA_OCURR'] = pd.to_datetime(df_datos['FECHA_OCURR'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_datos['CAN_LQ_COB'] = pd.to_numeric(df_datos['CAN_LQ_COB'], errors='coerce').astype('Int64')
df_datos['CAN_LQ_INC'] = pd.to_numeric(df_datos['CAN_LQ_INC'], errors='coerce').astype('Int64')
df_datos['CAN_LQ_ANU'] = pd.to_numeric(df_datos['CAN_LQ_ANU'], errors='coerce').astype('Int64')
df_datos['CAN_LQ_EMI'] = pd.to_numeric(df_datos['CAN_LQ_EMI'], errors='coerce').astype('Int64')
df_datos['PRIMA_COB'] = pd.to_numeric(df_datos['PRIMA_COB'], errors="coerce").astype('float64')
df_datos['PRIMA_INC_ANU'] = pd.to_numeric(df_datos['PRIMA_INC_ANU'], errors="coerce").astype('float64')
df_datos['PRIMA_BRUTA'] = pd.to_numeric(df_datos['PRIMA_BRUTA'], errors="coerce").astype('float64')
df_datos['MONTO_LA'] = pd.to_numeric(df_datos['MONTO_LA'], errors="coerce").astype('float64')
df_datos['MTOTOTRES'] = pd.to_numeric(df_datos['MTOTOTRES'], errors="coerce").astype('float64')
df_datos['MTOTOTAPROB'] = pd.to_numeric(df_datos['MTOTOTAPROB'], errors="coerce").astype('float64')
df_datos['MTOTOTPAGADO'] = pd.to_numeric(df_datos['MTOTOTPAGADO'], errors="coerce").astype('float64')
df_datos['MTOTOTPEND'] = pd.to_numeric(df_datos['MTOTOTPEND'], errors="coerce").astype('float64')



In [ ]:
df_ticket = df_datos[['FECHA_INI_ULT_COBERTURA', 'FECHA_FIN_ULT_COBERTURA', 'CALCULO_PRIMA']]

In [ ]:
df_ticket[df_ticket['CALCULO_PRIMA']=='A-ANUAL'].head(6)

,FECHA_INI_ULT_COBERTURA,FECHA_FIN_ULT_COBERTURA,CALCULO_PRIMA,FECHA_CALCULADA
17,2023-09-25,2024-09-25,A-ANUAL,2023-09-25 00:00:00
24,2024-01-16,2025-01-16,A-ANUAL,2024-01-16 00:00:00
26,2025-12-11,2026-12-11,A-ANUAL,2025-12-11 00:00:00
31,2023-12-29,2024-12-23,A-ANUAL,2023-12-23 00:00:00
33,2026-01-19,2026-11-20,A-ANUAL,2025-11-20 00:00:00
34,2025-06-10,2025-12-10,A-ANUAL,2024-12-10 00:00:00


In [ ]:
df_ticket[df_ticket['CALCULO_PRIMA']=='A-ANUAL'].head(6)

,FECHA_INI_ULT_COBERTURA,FECHA_FIN_ULT_COBERTURA,CALCULO_PRIMA
17,2023-09-25,2024-09-25,A-ANUAL
24,2024-01-16,2025-01-16,A-ANUAL
26,2025-12-11,2026-12-11,A-ANUAL
31,2023-12-23,2024-12-23,A-ANUAL
33,2025-11-20,2026-11-20,A-ANUAL
34,2024-12-10,2025-12-10,A-ANUAL


In [ ]:
print(type(df_ticket['FECHA_CALCULADA'].iloc[0]))

<class 'datetime.date'>


In [ ]:

def restar_fecha(fecha, periodicidad):
    if pd.isna(fecha):
        return fecha
    if periodicidad == 'M-MENSUAL':
        return fecha - pd.DateOffset(months=1)
    elif periodicidad in ('A-ANUAL','E-ESPECIAL'):
        return fecha - pd.DateOffset(years=1)
    else:
        return fecha

df_ticket['FECHA_INI_ULT_COBERTURA'] = df_ticket.apply(lambda r: restar_fecha(r['FECHA_FIN_ULT_COBERTURA'],
                                                                      r['CALCULO_PRIMA']), axis=1)

/tmp/ipykernel_2106/2813302441.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ticket['FECHA_INI_ULT_COBERTURA'] = df_ticket.apply(lambda r: restar_fecha(r['FECHA_FIN_ULT_COBERTURA'],


In [ ]:
df_datos['CALCULO_PRIMA'].value_counts()

,count
CALCULO_PRIMA,
-,152636
A-ANUAL,66523
E-ESPECIAL,8639
M-MENSUAL,7670


In [ ]:
df_datos.head(3)

,IDEPOL,CERT_BANCO_EXCEL,CERTIFICADO_BANCO,PRODUCTO,CODPROD,NUMPOL,CLIENTE,NUMCERT,STSCERT,FECHA_INICIO_VIGENCIA,...,STSSIN,FECHA_OCURR,MTOTOTRES,MTOTOTAPROB,MTOTOTPAGADO,MTOTOTPEND,DESCRAMO,TIPOVIG,CODMONEDA,CALCULO_PRIMA
0,6722450,00117794554000205568,00117794554000205568,NaN,2001,716888,JOSE ROBERTO ESPINOZA HUARACA,20448,ANU,2017-10-25,...,CER,2018-02-17,31.09,31.09,0.00,31.09,VEHICULOS (DAÑOS AL VEHICULO),L-LIBRE,USD,E-ESPECIAL
1,6722450,00117794524000456676,00117794524000456676,NaN,2001,716888,DANIEL ENRIQUE VILLAVICENCIO ROJAS,28515,ACT,2018-05-15,...,PAG,2018-07-04,211.26,211.26,211.26,0.00,VEHICULOS (DAÑOS AL VEHICULO),L-LIBRE,USD,E-ESPECIAL
2,6722450,00117794554000205568,00117794554000205568,NaN,2001,716888,JOSE ROBERTO ESPINOZA HUARACA,20448,ANU,2017-10-25,...,PAG,2018-07-25,1843.85,1843.85,1843.85,0.00,VEHICULOS (DAÑOS AL VEHICULO),L-LIBRE,USD,E-ESPECIAL
